# Đào tạo Model Nhận diện Chữ số viết tay (MNIST) với CNN
Notebook này chứa toàn bộ code cần thiết để huấn luyện mô hình mạng Neural tích chập (CNN) trên tập dữ liệu MNIST bằng TensorFlow/Keras. Bạn có thể mở trực tiếp trên Google Colab để tận dụng GPU để huấn luyện mô hình được nhanh hơn.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import BatchNormalization, Conv2D, Dense, Dropout, Flatten, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

print('TensorFlow version:', tf.__version__)

## 1. Tải và Tiền xử lý Dữ liệu

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print('Train shape:', x_train.shape)
print('Test shape:', x_test.shape)

## 2. Xây dựng Kiến trúc Model

In [ ]:
def build_model():
    return Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(10, activation='softmax'),
    ])

model = build_model()
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 3. Huấn luyện Model (Training)

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
)
datagen.fit(x_train)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=3, min_lr=1e-5
)
early_stopping = EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)
callbacks = [reduce_lr, early_stopping]

history = model.fit(
    datagen.flow(x_train, y_train, batch_size=128),
    epochs=20,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
)

## 4. Đánh giá và Tải Model về máy (Tùy chọn)

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f'Độ chính xác trên tập kiểm tra: {test_accuracy * 100:.2f}%')

MODEL_NAME = 'model.h5'
model.save(MODEL_NAME)
print(f'Mô hình đã được lưu thành công vào "{MODEL_NAME}"')

# Nếu bạn chỉ muốn tải về máy thủ công:
from google.colab import files
files.download(MODEL_NAME)

## 5. Tự động PUSH model lên Hugging Face (MLOps)
Sử dụng tính năng này để đẩy thẳng model lên Cloud. Sau đó ứng dụng Web của bạn có thể lấy model mới nhất bằng 1 cú click.

In [ ]:
!pip install huggingface_hub -q
from huggingface_hub import login, HfApi

# Yêu cầu quyền truy cập (Token Hugging Face của bạn)
login()

# Bạn có thể thay đổi repo_id theo ý muốn (ví dụ: username/mnist-model)
repo_id = input("Nhập Hugging Face Repo ID (Ví dụ: HungHiHung/mnist-cnn-model): ")

api = HfApi()
try:
    # Tạo repo tự động nếu chưa có
    api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")
    
    # Upload file model.h5
    print("Đang upload model lên Hugging Face...")
    api.upload_file(
        path_or_fileobj=MODEL_NAME,
        path_in_repo=MODEL_NAME,
        repo_id=repo_id,
        repo_type="model"
    )
    print(f"Thành công! Bạn có thể sử dụng repo ID '{repo_id}' trong trang Web để nạp model.")
except Exception as e:
    print(f"Lỗi trong quá trình upload: {str(e)}")